# Feature Engineering — Project Delay Prediction

Joins each task with project + assigned-subcontractor attributes and derives
pre-task features. EXCLUDES post-task leakage (actual_start_date, forecast_end_date,
schedule_variance_days, status, pct_complete).

**Reads:** `silver_tasks`, `silver_projects`, `silver_subcontractors`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
tk = spark.read.table('silver_tasks')
pr = spark.read.table('silver_projects')
sb = spark.read.table('silver_subcontractors')
print(f'tasks={tk.count():,} projects={pr.count():,} subcontractors={sb.count():,}')

required = {'task_id', 'project_id', 'assigned_subcontractor_id', 'is_delayed', 'planned_duration_days'}
missing = required - set(tk.columns)
if missing:
    raise ValueError(f'silver_tasks missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Subcontractor attrs (rename to avoid clash with project.region / task fields).
sub = sb.select(
    col('subcontractor_id').alias('assigned_subcontractor_id'),
    col('trade').alias('sub_trade'),
    col('rating').alias('sub_rating'),
    col('years_active').alias('sub_years'),
    when(col('accredited') == 'Y', lit(1)).otherwise(lit(0)).alias('sub_accredited_flag'),
)

# Join attributes + select pre-task features. EXCLUDE leakage.
ml_features = (
    tk.select(
        'task_id', 'project_id', 'assigned_subcontractor_id', 'task_name',
        'planned_duration_days',
        col('is_delayed').alias('had_delay'),
    )
    .join(pr.select('project_id', 'project_type',
                    col('region').alias('project_region'), 'budget'),
          'project_id', 'left')
    .join(sub, 'assigned_subcontractor_id', 'left')
    .na.fill(0)
    .na.fill('unknown', subset=['task_name', 'project_type', 'project_region', 'sub_trade'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_delay') == 1).count()
delay_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} delayed rows '
        f'({delay_rate:.2f}%). Check is_delayed typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | delay rate {delay_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')